Import the libraries used for API calls, environment settings, the chat interface, JSON handling, web requests, and reading PDF files.


In [34]:
from openai import OpenAI
import os
import gradio as gr
import json
from dotenv import load_dotenv
import requests
from pypdf import PdfReader

Load environment variables from the .env file and create the default OpenAI client.


In [35]:
load_dotenv(override=True)
openai= OpenAI()

Read the Pushover credentials from environment variables and check whether they appear to be present and correctly formatted.


In [36]:
pushover_user= os.getenv("PUSHOVER_USER")
pushover_token= os.getenv("PUSHOVER_TOKEN")
pushover_url= "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover User Key is set correctly.")
    else:
        print("Pushover user key is found but not correct")
else:
    print("Pushover key is not found")


if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover Token is set correctly.")
    else:
        print("Pushover token is found but not correct")
else:
    print("Pushover token is not found")    
    


Pushover User Key is set correctly.
Pushover Token is set correctly.


Define a helper that sends a message to Pushover as a notification.


In [37]:
def push(message):
    print(f"push : {message}")
    payload = { "user": pushover_user, "token": pushover_token, "message": message }
    response = requests.post(pushover_url, data=payload)

Send a short test notification to check the Pushover integration.


In [38]:
push("Hey")

push : Hey


SSLError: HTTPSConnectionPool(host='api.pushover.net', port=443): Max retries exceeded with url: /1/messages.json (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1010)')))

Define a tool function that sends a visitor?s contact details to you through a Pushover notification.


In [39]:
def record_user_details(email, name="not provided", phone="not provided", note="not provided"):
    push(f"User Details: Email: {email}, Name: {name}, Phone: {phone}, Note: {note}")
    return "OK"

Define a tool function for notifying you when the assistant cannot answer a visitor?s question.


In [40]:
def record_unknown_questions(question):
    push(f"Recording {question} asked that I couldn't answer")
    return "OK"

Describe the contact-details function in the JSON schema format used to expose it as an LLM tool.


In [41]:
record_user_details_json={

    "name":"record_user_details",
    "description":"Records user details and sends a notification via Pushover",
    "parameters":{
        "type":"object",
        "properties":{
            "email":{   
                    "type":"string",
                    "description":"The email address of the user"
                 },
            "name":{
                
                    "type":"string",
                    "description":"The name of the user"
                 },
            "phone":{
                    "type":"string",
                    "description":"The phone number of the user"
                 },
            "note":{
                    "type":"string",
                    "description":"Any additional notes about the user"
                 }
            },
            "required": ["email"]
        }
} 

Describe the unknown-question function in the JSON schema format used to expose it as an LLM tool.


In [ ]:
record_unknown_questions_json={
    "name":"record_unknown_questions",
    "description":"Records questions that the AI couldn't answer and sends a notification via Pushover",
    "parameters":{
        "type":"object",
        "properties":{
            "question":{
                "type":"string",
                "description":"The question that the AI couldn't answer"
            }
        },
        "required": ["question"]
    }
}

Collect both function schemas into the tools list that will be sent with model requests.


In [42]:
tools=[{"type":"function",  "function":record_user_details_json},
       {"type":"function", "function":record_unknown_questions_json}
      ]

Display the configured tool definitions so you can inspect their names and argument schemas.


In [43]:
tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Records user details and sends a notification via Pushover',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of the user'},
     'name': {'type': 'string', 'description': 'The name of the user'},
     'phone': {'type': 'string',
      'description': 'The phone number of the user'},
     'note': {'type': 'string',
      'description': 'Any additional notes about the user'}},
    'required': ['email']}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_questions',
   'description': "Records questions that the AI couldn't answer and sends a notification via Pushover",
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': "The question that the AI couldn't answer"}},
    'required': ['question']}}}]

Define a dispatcher that reads each requested tool call, runs the matching Python function, and formats its result for the model.


In [44]:
def handle_tool_calls_manual_if(tool_calls):
    results=[]
    for tool_call in tool_calls:
        tool_name= tool_call.function.name
        argument= json.loads(tool_call.function.arguments)
        print(f"Tool Name: {tool_name}")

        if tool_name=="record_user_details":
            result=record_user_details(**argument)
        elif tool_name=="record_unknown_questions":
            result=record_unknown_questions(**argument)

        results.append({"role":"tool", "content": json.dumps(result),"tool_call_id":tool_call.id})

    return results



Extract text from the LinkedIn PDF and read the prepared profile summary for use as context in the digital-twin prompt.


In [45]:
reader= PdfReader("twin/linkedin.pdf")
linkedin=""
for page in reader.pages:
    text= page.extract_text() + "\n"
    if text:
        linkedin+=text

with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()


Build the system prompt that defines the digital twin?s role, supplies profile context, and explains when to use the available tools.


In [46]:
system_prompt= f"""

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Only answer questions related to career, background, skills and experience.
If the user asks about something unrelated, then steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

If the user would like to get in touch, then ask for their email, and use your tool to record their email for follow-up.

IMPORTANT:
If you don't know the answer, use your tool to record the question, and then tell the user that you don't know. Never make up an answer.


"""

Create a Gemini client using the configured API key and base URL, and choose the model used for chat requests.


In [47]:
gemini_api_key= os.getenv("GEMINI_API_KEY1")
gemini_base_url= os.getenv("GEMINI_BASE_URL")
gemini= OpenAI(api_key=gemini_api_key,base_url=gemini_base_url)
gemini_model="gemini-3.6-flash"

Define the chat handler that sends conversation history to Gemini, processes requested tool calls, and returns the final reply.


In [48]:
def chat(message, history):
    messages=[{"role":"system", "content":system_prompt}]+history+[{"role":"user", "content":message}]
    response=gemini.chat.completions.create(
        model=gemini_model,
        messages=messages,
        tools=tools
    )
    while response.choices[0].finish_reason=="tool_call":
        message= response.choices[0].message
        tool_Calls= message.tool_calls
        results= handle_tool_calls_manual_if(tool_Calls)
        messages.append(message)
        messages.extends(results)
        response=gemini.chat.completions.create(
            model=gemini_model,
            messages=messages,
            tools=tools 
        )
    return response.choices[0].message.content


        

Launch the Gradio chat interface in a browser so you can interact with the digital twin.


In [ ]:
theme = gr.themes.Soft(
    primary_hue="emerald",
    secondary_hue="emerald",
    neutral_hue="slate",
    radius_size="lg",
)

css = """
.gradio-container {
    max-width: 860px !important;
    margin: 0 auto !important;
    background: #ffffff !important;
}
.chat-header {
    max-width: 760px;
    margin: 0 auto !important;
    padding: 2.5rem 1rem 1.25rem !important;
    background: transparent !important;
    border: 0 !important;
    box-shadow: none !important;
    text-align: center;
}
.chat-header h1 {
    margin: 0 0 .5rem;
    color: #202123;
    font-size: clamp(1.8rem, 5vw, 2.35rem);
    font-weight: 600;
    letter-spacing: -.035em;
}
.chat-header p { margin: 0; color: #6b7280; font-size: 1rem; }
.hero-kicker { display: none; }
#karan-chatbot {
    max-width: 760px;
    margin: 0 auto;
    border: 1px solid #e5e7eb !important;
    border-radius: 1.25rem !important;
    background: #ffffff !important;
    box-shadow: 0 8px 28px rgba(17, 24, 39, .06) !important;
    overflow: hidden;
    transition: border-color .2s ease, box-shadow .2s ease;
}
#karan-chatbot [data-testid="user"] {
    box-sizing: border-box;
    padding: .9rem 1.15rem !important;
    border: 1px solid #bfdbfe !important;
    border-radius: 1.25rem 1.25rem .35rem 1.25rem !important;
    background: #eff6ff !important;
    color: #1e3a8a !important;
    box-shadow: 0 2px 8px rgba(37, 99, 235, .08);
}
#karan-chatbot [data-testid="user"] * {
    color: #1e3a8a !important;
    -webkit-text-fill-color: #1e3a8a !important;
}
#karan-chatbot [data-testid="bot"] {
    box-sizing: border-box;
    padding: .9rem 1.15rem !important;
    border: 2px solid #86efac !important;
    border-left: 5px solid #10a37f !important;
    border-radius: 1.25rem 1.25rem 1.25rem .35rem !important;
    background: #f0fdf4 !important;
    color: #14532d !important;
    -webkit-text-fill-color: #14532d !important;
    box-shadow: 0 2px 8px rgba(16, 163, 127, .07);
}
#karan-chatbot [data-testid="bot"] * {
    color: #14532d !important;
    -webkit-text-fill-color: #14532d !important;
}
#karan-input {
    max-width: 760px;
    margin: .75rem auto 1rem;
    border: 1px solid #bfdbfe !important;
    border-radius: 1.5rem !important;
    background: #eff6ff !important;
    box-shadow: 0 2px 10px rgba(37, 99, 235, .08) !important;
    transition: border-color .2s ease, box-shadow .2s ease;
}
#karan-input textarea,
#karan-input textarea:focus {
    background: #eff6ff !important;
    color: #1e3a8a !important;
    -webkit-text-fill-color: #1e3a8a !important;
    caret-color: #2563eb !important;
}
#karan-input textarea::selection {
    background: #bfdbfe !important;
    color: #1e3a8a !important;
}
#karan-input textarea::placeholder {
    color: #64748b !important;
    -webkit-text-fill-color: #64748b !important;
    opacity: 1;
}
#karan-input:focus-within {
    border-color: #93c5fd !important;
    box-shadow: 0 0 0 3px rgba(59, 130, 246, .12), 0 2px 10px rgba(37, 99, 235, .08) !important;
}
button { transition: background-color .15s ease, transform .15s ease !important; }
button:hover { transform: translateY(-1px); }
@media (max-width: 640px) {
    .chat-header { padding: 1.5rem .75rem .75rem !important; }
    #karan-input { margin: .5rem .25rem .75rem; }
}
@media (prefers-color-scheme: dark) {
    #karan-chatbot [data-testid="user"] { background: #eff6ff !important; color: #1e3a8a !important; }
    #karan-chatbot [data-testid="bot"] { background: #f0fdf4 !important; color: #14532d !important; }
    #karan-chatbot [data-testid="bot"] * { color: #14532d !important; -webkit-text-fill-color: #14532d !important; }
    #karan-input { background: #eff6ff !important; }
    #karan-input textarea { background: #eff6ff !important; color: #1e3a8a !important; -webkit-text-fill-color: #1e3a8a !important; }
}
@media (prefers-reduced-motion: reduce) {
    *, *::before, *::after { scroll-behavior: auto !important; transition-duration: .01ms !important; }
}
"""

with gr.Blocks(title="Karan Pratap Singh | AI Assistant") as demo:
    gr.Markdown(
        """
        # Karan Pratap Singh
        Ask me about my background, skills, and experience.
        """,
        elem_classes=["chat-header"],
    )
    gr.ChatInterface(
        fn=chat,
        chatbot=gr.Chatbot(
            height=520,
            layout="panel",
            buttons=["copy_all"],
            elem_id="karan-chatbot",
        ),
        textbox=gr.Textbox(
            placeholder="Message Karan...",
            container=False,
            elem_id="karan-input",
        ),
        examples=[
            "Tell me about your professional background.",
            "What test automation tools have you worked with?",
            "What are your interests in AI and automation?",
        ],
        flagging_mode="never",
    )

demo.launch(inbrowser=True, theme=theme, css=css)
